# London Crime Analysis

### Environment Setup

In [1]:
# Standard library
from pathlib import Path

# Third-party
import pandas as pd
import numpy as np

# Project — triggers config + logging initialization
from london_crime.config import config
from london_crime.logging_config import get_logger

# Notebook display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

logger = get_logger("notebooks.01_exploration")
logger.info("Notebook 01_exploration started")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Project root: {config.paths.project_root}")

2026-09-13 14:30:29,987 | london_crime.config | INFO | Configuration loaded from: ..\config.yaml
2026-09-13 14:30:29,988 | notebooks.01_exploration | INFO | Notebook 01_exploration started
Pandas: 3.0.5
NumPy: 2.5.3
Project root: D:\ML\Portfolio\Projects\london-crime-analysis


### Load the Combined CSV

In [2]:
raw_csv_path = config.paths.raw_data_dir / config.data["raw_filename"]
logger.info(f"Loading: {raw_csv_path.relative_to(config.paths.project_root)}")

# Load with everything as strings — preserve the raw mess
df = pd.read_csv(raw_csv_path, dtype=str, keep_default_na=False)

logger.info(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

2026-09-13 14:32:04,125 | notebooks.01_exploration | INFO | Loading: data\raw\london_crimes.csv
2026-09-13 14:32:06,332 | notebooks.01_exploration | INFO | Loaded 1,140,416 rows × 13 columns
Shape: (1140416, 13)
Rows: 1,140,416
Columns: 13


### First Look

In [3]:
# First 5 rows, all columns
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,source_file
0,4af53e255f9f3e69d1a295a288fe02488234842b18020e0b9c6ec10452724297,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.549537,50.815808,On or near Peel Close,E01031427,Arun 004A,Violence and sexual offences,Status update unavailable,,2025-01-metropolitan-street.csv
1,790fd102d1e83b2239b5a365ab1fccbe028e6b1c9333ae96ddd1498dd04074e5,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.212745,51.409357,On or near Supermarket,E01003423,Merton 011E,Violence and sexual offences,Unable to prosecute suspect,,2025-01-metropolitan-street.csv
2,600375a5149e92bdf80d3be0db0b31ce9b8df32091f7ab6d0d2f2058489e772a,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.211590,51.410949,On or near Richmond Avenue,E01003423,Merton 011E,Vehicle crime,Unable to prosecute suspect,,2025-01-metropolitan-street.csv
3,087d78d903f332ba60ae35ddfdd052edff68d80cd42d1f701dad07e842c0d858,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.207944,51.410039,On or near Watery Lane,E01003423,Merton 011E,Vehicle crime,Investigation complete; no suspect identified,,2025-01-metropolitan-street.csv
4,45a4272ac23f592c44c3b24f3008d308985d3f6ed3e664e861c64d0f5292ffd4,2025-01,Metropolitan Police Service,Metropolitan Police Service,-0.212700,51.410885,On or near Chatsworth Avenue,E01003423,Merton 011E,Vehicle crime,Investigation complete; no suspect identified,,2025-01-metropolitan-street.csv


### Per-Column Stats

In [5]:
# For each column: how many empty strings?
empty_counts = (df == "").sum().sort_values(ascending=False)
empty_pct = (empty_counts / len(df) * 100).round(2)

summary = pd.DataFrame({
    "empty_count": empty_counts,
    "empty_pct": empty_pct,
})
summary

,empty_count,empty_pct
Context,1140416,100.0
Last outcome category,234967,20.6
Crime ID,234967,20.6
Month,0,0.0
Reported by,0,0.0
Longitude,0,0.0
Falls within,0,0.0
Latitude,0,0.0
Location,0,0.0
LSOA name,0,0.0


### Basic Info

In [7]:
# Confirm: empty strings vs actual NaN
print("Total rows:", len(df))
print("\nEmpty string counts (the real 'missing'):")
print((df == "").sum().sort_values(ascending=False))

# Check if Crime ID and Last outcome category emptiness overlap
both_empty = ((df["Crime ID"] == "") & (df["Last outcome category"] == "")).sum()
print(f"\nRows where BOTH Crime ID and Last outcome are empty: {both_empty:,}")
print(f"Rows where ONLY Crime ID is empty: {((df['Crime ID'] == '') & (df['Last outcome category'] != '')).sum():,}")
print(f"Rows where ONLY Last outcome is empty: {((df['Crime ID'] != '') & (df['Last outcome category'] == '')).sum():,}")

Total rows: 1140416

Empty string counts (the real 'missing'):
Context                  1140416
Last outcome category     234967
Crime ID                  234967
Month                          0
Reported by                    0
Longitude                      0
Falls within                   0
Latitude                       0
Location                       0
LSOA name                      0
LSOA code                      0
Crime type                     0
source_file                    0
dtype: int64

Rows where BOTH Crime ID and Last outcome are empty: 234,967
Rows where ONLY Crime ID is empty: 0
Rows where ONLY Last outcome is empty: 0


In [6]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1140416 entries, 0 to 1140415
Data columns (total 13 columns):
 #   Column                 Non-Null Count    Dtype
---  ------                 --------------    -----
 0   Crime ID               1140416 non-null  str  
 1   Month                  1140416 non-null  str  
 2   Reported by            1140416 non-null  str  
 3   Falls within           1140416 non-null  str  
 4   Longitude              1140416 non-null  str  
 5   Latitude               1140416 non-null  str  
 6   Location               1140416 non-null  str  
 7   LSOA code              1140416 non-null  str  
 8   LSOA name              1140416 non-null  str  
 9   Crime type             1140416 non-null  str  
 10  Last outcome category  1140416 non-null  str  
 11  Context                1140416 non-null  str  
 12  source_file            1140416 non-null  str  
dtypes: str(13)
memory usage: 970.9 MB
